# L4b: Single Asset Geometric Brownian Motion Models
In this lecture, we introduce our first continuous-time model for asset prices: geometric Brownian motion (GBM). The model is widely used because it has an exact solution, can be simulated exactly on a time grid, and gives a closed-form probability for the terminal trade rule developed in L4a.

> __Learning Objectives:__
>
> By the end of this lecture, you will be able to:
>
> * **Model a share price with GBM:** State the stochastic differential equation, interpret its Wiener-process noise, derive the lognormal solution, and simulate exact grid-point paths.
> * **Estimate and interpret GBM parameters:** Estimate mean growth and volatility from one-step growth rates, recover the arithmetic drift, and explain what a log-price regression does and does not establish.
> * **Evaluate and criticize the model:** Compare GBM with the stylized facts and compute the probability that a scheduled long position clears a terminal return target.

Let's get started!

___

## Examples
Two examples connect the lecture's formulas to data and decisions:

- [▶ Estimate a single-asset GBM](CHEME-5660-L4b-Example-Parameters-SAGBM-Fall-2026.ipynb). Estimate mean growth and volatility, recover the arithmetic drift, simulate paths, and compare the model with an observed price trajectory.
- [▶ Evaluate an NPV-based GBM trade rule](CHEME-5660-L4b-Example-GBM-NPV-TradeRule-Fall-2026.ipynb). Compute and visualize the probability that a scheduled long position clears a target present-value return.

Optional extensions are collected near the end of the lecture.

___

## From Lattices to Continuous Prices
Quantitative trading firms such as [Jane Street](https://www.janestreet.com) use probability models to connect uncertain price changes with trading and execution decisions. Our concern here is the model, not a comparison among firms: how can the finite lattice of L4a become a continuous distribution in continuous time?

Adding branches by itself is not enough. The branch factors, probabilities, and time step must be scaled together so that the one-step mean and variance approach finite limits. Under the scaling developed in the optional lattice-limit notebook, the terminal log price becomes Gaussian and the price becomes lognormal. Geometric Brownian motion is that continuous limiting model.

The [N-ary lattice example](../L4a/CHEME-5660-L4a-Example-N-Ary-Lattice-Fall-2026.ipynb) shows how more branches refine an empirical one-step distribution while preserving recombination. Today's lecture replaces the discrete time grid with continuous time and specifies the limiting random process directly.

The optional [lattice-to-GBM notebook](advanced/lattice-limit/CHEME-5660-L4b-Advanced-LatticeToGBM-Fall-2026.ipynb) makes the limiting argument numerical and explicit. For the main lecture, we begin from the resulting stochastic differential equation.

___

## Single Asset Geometric Brownian Motion (GBM)
Geometric Brownian motion (GBM) is a continuous-time stochastic price process whose drift and diffusion coefficients are both proportional to the current price $S(t)$. Its log price is a Brownian motion with drift. Over a horizon $0\leq t\leq T$, the price obeys:
$$
\begin{align*}
\frac{dS\left(t\right)}{S(t)} &= \mu\,{dt}+\sigma\,{dW(t)}.\\
\end{align*}
$$
Here, $\mu\in\mathbb{R}$ (units: inverse years) is the arithmetic drift in the GBM price SDE, $\sigma>0$ (units: inverse years to the one-half power) is a constant __volatility__ parameter, and $dW(t)$ is the increment of a Wiener process. We reserve $g$ for an observed continuously compounded growth rate. The drift $\mu$ and a growth rate $g$ share inverse-time units, but they are different quantities; we make the relationship precise below. First, what is a Wiener process?

> __Wiener process:__
> 
> A Wiener process $\{W(t):0\leq t\leq T\}$ is a continuous, real-valued stochastic process with the following properties:
>
> * The process starts at zero: $W(0)=0$ with probability one.
> * The increments $W(t_{1})-W(t_{0}),\ldots,W(t_{k})-W(t_{k-1})$ are independent for any $0\leq t_{0}<t_{1}<\cdots<t_{k}\leq T$.
> * For $0\leq s<t\leq T$, the increment law is $W(t)-W(s)\sim\mathcal N(0,t-s)$. Equivalently, the increment has the same distribution as $\sqrt{t-s}Z$, where $Z\sim\mathcal N(0,1)$.

The [GBM stochastic differential equation has an exact solution derived with Itô calculus](CHEME-5660-L4b-GBM-Solution-Derivation-Fall-2026.ipynb). Take $t_{0}=0$ and let $S_{0}$ be the current share price. The share price at a fixed future time $T$ is given by:
$$
\boxed{
\begin{align*}
S_{T} &= S_{0}\;\exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)T + \sigma\sqrt{T}\;Z\Biggr]\qquad Z\sim\mathcal{N}(0,1).\\
\end{align*}}
$$
Thus, the log price ratio $\ln(S_{T}/S_{0})$ is normally distributed with mean $(\mu-\sigma^{2}/2)T$ and variance $\sigma^{2}T$, and the price $S_{T}$ itself is lognormally distributed and strictly positive. The expectation and variance of $S_{T}$ are given by:
$$
\begin{align*}
\mathbb{E}\left(S_{T}\right) &= S_{0}\;e^{\mu T},\\
\text{Var}\left(S_{T}\right) &= S_{0}^{2}\;e^{2\mu T}\left[e^{\sigma^{2}T} - 1\right].
\end{align*}
$$
For any horizon $T>0$, these expressions show that the expected price grows exponentially at the arithmetic drift $\mu$, while the median price $S_{0}e^{(\mu-\sigma^{2}/2)T}$ grows at the smaller rate $\mu-\sigma^{2}/2$. The gap between the two is the __half-variance correction__: the lognormal distribution is right skewed, so its mean sits above its median.

### Discrete-Time GBM Model
Market data record prices at discrete times, such as once per trading day. Define the grid $t_{j}=j\Delta t$ for $j=0,1,\ldots,N$, where $\Delta t$ is measured in years and $T=N\Delta t$. Applying the fixed-horizon solution over one step gives the __one-step transition__:
$$
\boxed{
\begin{align*}
S_{t_{j}} &= S_{t_{j-1}}\;\exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)\Delta{t} + \sigma\sqrt{\Delta{t}}\;Z_{j}\Biggr]\qquad{j=1,2,\dots,N},\\
\end{align*}}
$$
where the shocks $Z_{1},Z_{2},\dots,Z_{N}$ are __independent__ standard normal random variables (they come from independent Wiener increments). For constant $\mu$ and $\sigma$ this transition is exact at the grid points (it says nothing about the path between them); it is not a numerical time-stepping approximation. Alternatively, we can jump directly from $S_{0}$ to grid point $t_{j}$ using the fixed-horizon solution with $T = t_{j}$:
$$
\begin{align*}
S_{t_{j}} &= S_{0}\;\exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)t_{j} + \sigma\sqrt{t_{j}}\;\tilde{Z}_{j}\Biggr]\qquad\text{where}\quad\tilde{Z}_{j} = \frac{1}{\sqrt{j}}\sum_{i=1}^{j}Z_{i}\quad{j=1,2,\dots,N}.\\
\end{align*}
$$
The cumulative shock $\tilde{Z}_{j}$ is standard normal at each fixed $j$, but $\tilde{Z}_{j}$ and $\tilde{Z}_{k}$ share their first $\min(j,k)$ increments, so they are __not__ independent: their correlation is $\sqrt{t_{\min(j,k)}/t_{\max(j,k)}}$. Use the cumulative form when you need the distribution at one horizon; use the one-step transition, with fresh independent shocks, when you need a price path.

### Monte Carlo Simulation of GBM
We can use the one-step transition to simulate many possible price paths, i.e., possible alternative future prices, using Monte Carlo simulation. 

__Monte Carlo idea.__ Generate independent shocks $Z_j$ for every step and every path, propagate the exact one-step transition, and summarize the resulting distribution of price paths.

Let's look at some pseudo-code for the Monte Carlo simulation of the GBM model.

__Initialize:__ Given the initial price $S_{0}$, arithmetic drift $\mu$, volatility $\sigma$, time step $\Delta{t}$, number of steps $N$, and number of sample paths $M$. Initialize an array $\mathbf{S}\in\mathbb{R}^{(N+1)\times M}$ to hold the simulated prices, one path per column, with rows indexed by $j = 0,1,\dots,N$.

For each sample path (each column of $\mathbf{S}$) __do:__
1. Set the price in the first row (grid point $j = 0$) to the initial price $S_{0}$.
2. For each time step $j = 1$ to $N$ __do:__
   - a. Generate a random sample from the standard normal distribution: $Z_{j} \sim \mathcal{N}(0,1)$
   - b. Compute the price at grid point $t_{j}$ from the price at $t_{j-1}$ using the one-step transition:
   $$
   S_{t_{j}} \gets S_{t_{j-1}} \cdot \exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)\Delta{t} + \sigma\sqrt{\Delta{t}}\;{Z_{j}}\Biggr].
   $$

Every shock is a fresh independent draw, for every step and for every path. This process generates $M$ different price paths (columns) of $N$ steps each, every one a possible future trajectory of the asset price under the GBM model. We've implemented this logic in [the `sample(...)` function](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/equity/#VLQuantitativeFinancePackage.sample-Tuple{MyGeometricBrownianMotionEquityModel,%20NamedTuple}).

___


## Estimation of GBM Parameters
The GBM drift and volatility control different features of the price distribution. To see how each parameter appears in data, divide the one-step transition by $S_{t_{j-1}}$, take the natural logarithm, and divide by the time step. This gives the __one-step growth rate__ $g_{j}$ over the interval $t_{j-1}\rightarrow t_{j}$, the same continuously compounded growth rate computed from price data in L3a:
$$
\boxed{
\begin{align*}
g_{j} &\equiv \left(\frac{1}{\Delta{t}}\right)\ln\left(\frac{S_{t_{j}}}{S_{t_{j-1}}}\right) = \underbrace{\left(\mu-\frac{\sigma^{2}}{2}\right)}_{\text{mean growth}\;\bar{g}} + \underbrace{\frac{\sigma}{\sqrt{\Delta{t}}}}_{\text{growth-rate std}}\;Z_{j}\qquad{j=1,2,\dots,N}.\\
\end{align*}}
$$
Because the shocks $Z_{j}$ are independent standard normals, the one-step growth rates over equally spaced, non-overlapping steps under constant-parameter GBM are independent and normally distributed with mean $\bar{g}\equiv\mu-\sigma^{2}/2$ and variance $\sigma^{2}/\Delta{t}$. Every estimator in this section falls out of this display. A word on notation: $\bar g$ is the population mean growth rate (a model parameter), $g^{\prime}$ is L3a's symbol for the sample mean of a growth-rate series, and $\widehat{\bar g}$ denotes an estimate of $\bar g$ by whichever method we are discussing.

> __Parameters:__
> 
> * __Drift versus growth:__ In $dS/S=\mu\,dt+\sigma\,dW$, $\mu$ is the arithmetic SDE drift. The __mean growth rate__ $\bar{g}=\mathbb{E}[g_{j}]=\mu-\sigma^{2}/2$ (units: inverse years) is what a growth-rate series measures. The two differ by the half-variance correction. Historical estimation makes neither quantity a guaranteed reward.
> * __Volatility:__ The parameter $\sigma\in\mathbb{R}_{>0}$ (units: inverse years to the one-half power) sets the standard deviation of the one-step growth rate, $\sigma/\sqrt{\Delta{t}}$, and of the one-step log return $g_{j}\Delta{t}$, which is $\sigma\sqrt{\Delta{t}}$. It measures dispersion from the model's Brownian shock, not every source of investment risk.

We estimate $\bar{g}$ and $\sigma$ from a growth-rate series and then recover $\mu$ from the half-variance correction. Let's start with volatility.

### Volatility
Suppose we have price observations $\left\{S_{t_{0}}, S_{t_{1}}, \ldots, S_{t_{N}}\right\}$ on the grid $t_{j} = j\Delta{t}$, so the sample spans $T = N\Delta{t}$ years. From these $N+1$ prices, we construct the $N$ one-step growth rates $\left\{g_{1},g_{2},\ldots,g_{N}\right\}$. The sample mean $g^{\prime}$ and the sample standard deviation $\sigma_{g}$ of the growth-rate series (our risk measure from L3a) are given by:
$$
\begin{align*}
g^{\prime} &= \frac{1}{N}\sum_{j=1}^{N}g_{j},\\
\sigma_{g} &= \underbrace{\sqrt{\frac{1}{N - 1}\sum_{j=1}^{N}\left(g_j - g^{\prime}\right)^2}}_{\text{volatility of the growth rate}}.\\
\end{align*}
$$
The denominator $N-1$ is the degrees of freedom of the sample standard deviation: $N$ growth observations less the one mean estimated from them. (In L3a we wrote the same estimator for $T$ price observations; here the sample has $N$ steps so that its span is $T=N\Delta t$.) We compute the standard deviation with [the `std(...)` function from Julia's `Statistics` standard library](https://docs.julialang.org/en/v1/stdlib/Statistics/#Statistics.std).

However, $\sigma_{g}$ is __not__ the GBM volatility $\sigma$. From the growth-rate display, the standard deviation of $g_{j}$ is $\sigma/\sqrt{\Delta{t}}$, so $\sigma_{g}$ estimates $\sigma/\sqrt{\Delta{t}}$, and the volatility estimate is given by:
$$
\boxed{
\begin{align*}
\hat{\sigma} &= \sigma_{g}\sqrt{\Delta{t}}.\\
\end{align*}}
$$
For daily data, $\Delta{t} = 1/252$ years, so the standard deviation of the annualized daily growth rates is divided by $\sqrt{252}$. Equivalently, the standard deviation of the daily log returns $g_{j}\Delta{t}$ is multiplied by $\sqrt{252}$. The companion example implements the first form, $\hat\sigma=\operatorname{std}(g)\sqrt{\Delta t}$; both forms give the same result when computed from the same returns.

__Drift.__ Next, estimate the mean growth rate $\bar{g}$ (units: inverse years). Under GBM, the arithmetic SDE drift is $\mu=\bar g+\sigma^2/2$, so once we have an estimate $\widehat{\bar g}$ of the mean growth rate, the drift estimate is $\hat{\mu}=\widehat{\bar g}+\hat{\sigma}^2/2$. The mean growth is small relative to one-step growth-rate noise, so it is usually estimated less precisely than volatility. There are two common ways to estimate $\bar{g}$:

> __Estimating the mean growth rate $\bar g$:__
>
> * __Method 1: Average growth rate.__ Compute the growth rates and take their sample mean, $\widehat{\bar g} = g^{\prime}$. Under GBM this is also the maximum likelihood estimate of $\bar g$.
> * __Method 2: Linear regression.__ Regress the log price on time. The slope estimates $\bar g$, but the log-price errors have Brownian covariance rather than independent noise. Fitting an intercept and slope projects those errors into residuals that remain dependent and heteroskedastic. The slope is descriptive, while ordinary iid regression standard errors and residual tests are not calibrated for this setting.
> 
> The next subsection develops the linear-regression estimate and its limitations.

### Linear Regression
Let's assume we have a price dataset for some firm, collected every day for some time period, on the grid $t_{j} = j\Delta{t}$: $\left\{S_{t_{0}}, S_{t_{1}}, \ldots, S_{t_{N}}\right\}$. Taking the natural logarithm of the cumulative GBM solution at grid point $t_{j}$ gives:
$$
\begin{align*}
\ln(S_{t_{j}}) &= \ln(S_{0}) + \bar g\;{t_{j}} + \sigma\;{W(t_{j})}\qquad{j=0,1,\ldots, N}.\\
\end{align*}
$$
The expected value of the noise term is zero, so the __expected__ log price is a straight line in time with intercept $\ln(S_{0})$ and slope $\bar g$. That is why regressing the log price on time estimates the mean growth rate: the regression recovers the line, and the Wiener term $\sigma{W(t_{j})}$ plays the role of the error term. Notice what that error term is. The raw error vector has Brownian covariance $\operatorname{Cov}[W(t_i),W(t_j)]=\min(t_i,t_j)$, so neighboring errors are strongly correlated and their marginal variance grows with time. After fitting the intercept and slope, the residuals are a projection of that error vector: they remain dependent and heteroskedastic, but their variance need not grow monotonically along the sample. Ordinary least squares still gives a descriptive slope, while textbook iid standard errors, confidence intervals, and residual-normality checks are not calibrated for this covariance structure. (The companion example displays an Anderson-Darling statistic for the residuals, but its iid p-value is not inferentially valid for this dependent sample.)

To set up the regression, we construct an overdetermined system of equations from the $N+1$ log prices:
$$
\begin{align*}
\hat{\mathbf{X}}\mathbf{\theta} + \epsilon &= \mathbf{y},\\
\end{align*}
$$
where $\mathbf{\theta}$ contains the model parameters, the intercept $\ln(S_{0})$ and the slope $\bar g$, and $\epsilon$ is the error vector. We estimate the intercept along with the slope, rather than pinning it to the observed $\ln(S_{0})$, so that the fitted line passes through the middle of the data instead of through the first day. The (augmented) design matrix $\hat{\mathbf{X}}$ holds a column of ones for the intercept and the grid times:
$$
\begin{align*}
\hat{\mathbf{X}} &= \begin{bmatrix}
1 & t_{0} \\
1 & t_{1} \\
\vdots & \vdots \\
1 & t_{N}
\end{bmatrix},\\
\end{align*}
$$
while the observation vector $\mathbf{y}$ holds the log prices:
$$
\begin{align*}
\mathbf{y} &= \begin{bmatrix}
\ln(S_{t_{0}}) \\
\ln(S_{t_{1}}) \\
\vdots \\
\ln(S_{t_{N}})
\end{bmatrix}.\\
\end{align*}
$$
When $\hat{\mathbf X}$ has full column rank, the least-squares parameter estimate has the normal-equation solution:
$$
\boxed{
\begin{align*}
\hat{\mathbf{\theta}} &= (\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}})^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}.\quad\blacksquare\\
\end{align*}}
$$
The second component of $\hat{\mathbf{\theta}}$ is the regression estimate $\widehat{\bar g}$ of the mean growth rate.

The companion [single-asset GBM parameter example](CHEME-5660-L4b-Example-Parameters-SAGBM-Fall-2026.ipynb) estimates mean growth and volatility for each firm in a dataset, recovers the GBM drift, simulates sample paths reproducibly, and compares the model with a fixed firm's observed price trajectory.

___


## Stylized Facts
A useful price model should be judged against empirical regularities rather than accepted because it is convenient. For equally spaced, non-overlapping intervals, constant-parameter GBM implies:
$$
g_j=\bar g+\frac{\sigma}{\sqrt{\Delta t}}Z_j,
\qquad Z_j\overset{\mathrm{iid}}{\sim}\mathcal N(0,1),
$$
where $g_j=(1/\Delta t)\ln(S_{t_j}/S_{t_{j-1}})$. Over a horizon $T$, the average growth rate satisfies:
$$
g_{T,0}=\frac{1}{T}\ln\left(\frac{S_T}{S_0}\right)
=\bar g+\frac{\sigma}{\sqrt T}Z.
$$
Its variance decreases as $1/T$, even though the variance of the cumulative log return $Tg_{T,0}$ grows as $\sigma^2T$.

> __Which stylized facts can constant-parameter GBM reproduce?__
>
> - **Heavy-tailed growth rates:** The GBM model fails this fact. Every one-step and horizon-average growth rate is Gaussian, so the model assigns too little probability to extreme empirical moves. A lognormal *price* tail is not a heavy-tailed *growth-rate* distribution.
> - **Absence of linear autocorrelation:** The GBM model reproduces this fact for non-overlapping one-step growth rates. Independence gives:
>   $$
>   \operatorname{Cov}(g_j,g_{j+\tau})
>   =\frac{\sigma^2}{\Delta t}\operatorname{Cov}(Z_j,Z_{j+\tau})=0,
>   \qquad \tau\ge1.
>   $$
> - **Volatility clustering:** The GBM model fails this fact. The constant parameter $\sigma$ gives no mechanism for alternating high- and low-volatility periods or for dependence in squared growth rates.
>
> The GBM model is therefore a tractable baseline, not a complete empirical model: it captures the lack of linear predictability while missing tail risk and time-varying volatility.

That diagnosis tells us how to use the model responsibly. We can exploit its closed-form distribution for a terminal decision while keeping the failed assumptions visible.

___

## GBM Trade Rule
Let's develop an expression that allows us to compute the probability that a trade clears a target return, based on the net present value (NPV) of the trade, when the share price follows GBM.

Suppose we purchase $n_0>0$ shares of ticker `XYZ` at time $0$ for $S_0$ USD/share and sell all shares at $T=N\Delta t$ for $S_T$ USD/share, where $N\ge1$ and $\Delta t$ is measured in years.

This trade is a __long position__ because its terminal sale value increases with $S_T$. In the frictionless, no-dividend baseline, an undiscounted gain requires $S_T>S_0$, while a positive discounted NPV requires $S_T>S_0e^{g_yT}$. We represent the trade by two cash flows: the purchase at $t=0$ and the sale at $t=T$. From the investor's perspective, its net present value is given by:
$$
\begin{align*}
\texttt{NPV}(g_y, T) &= \underbrace{-n_{0}\;{S_{0}}}_{\text{entry (now)}} + \underbrace{n_{0}\;{S_{T}}\;\mathcal{D}_{T,0}^{-1}(g_y)}_{\text{exit (future)}}\quad\Longrightarrow\text{divide by initial investment}\;n_{0}\;{S_{0}}\\
\frac{\texttt{NPV}(g_y, T)}{n_{0}\;{S_{0}}} &= \frac{S_{T}\;\mathcal{D}_{T,0}^{-1}(g_y) - S_{0}}{S_{0}}\\
\underbrace{\frac{\texttt{NPV}(g_y, T)}{n_{0}\;{S_{0}}}}_{\text{fractional return}\;\rho_{T}} &= \left(\frac{S_{T}}{S_{0}}\right)\;\mathcal{D}_{T,0}^{-1}(g_y) - 1.\quad\blacksquare
\end{align*}
$$
Following [L1b](../../week-1/L1b/CHEME-5660-L1b-Lecture-TimeValueMoney-Fall-2026.ipynb) and L4a, $g_y$ denotes the continuously compounded annual growth rate associated with the selected benchmark yield $y$, $\mathcal{D}_{T,0}(g_y)=e^{g_yT}$ is its accumulation factor, and $\mathcal{D}^{-1}_{T,0}(g_y)=e^{-g_yT}$ discounts the terminal sale value to time 0. We call the left-hand side, the NPV per dollar invested, the scaled NPV $\rho_{T}$; it is a dimensionless present-value return on the initial share cost.

Last lecture, we used a binomial lattice model for $S_{T}$. Let's now assume that the share price $S_{T}$ follows a GBM model. Substituting the GBM solution for $S_{T}$ into the scaled NPV expression gives:
$$
\boxed{
\begin{align*}
\rho_{T} = \frac{\texttt{NPV}(g_y, T)}{n_{0}{S_{0}}} &= \exp\Biggl[\left(\mu-\frac{\sigma^{2}}{2}\right)T + \sigma\sqrt{T}\;Z\Biggr]\;e^{-g_yT} - 1\qquad Z\sim\mathcal{N}(0,1).\quad\blacksquare\\
\end{align*}}
$$
If we hold the $n_{0}$ shares for only a __short time__, e.g., a few days or a few weeks, then for realistic choices of the benchmark rate $|g_y|T$ is small, $e^{-g_yT}\approx1$, and $\rho_{T}\approx S_{T}/S_{0}-1$ is approximately the fractional change in share price. For a longer holding period, or whenever $|g_y|T$ is not small, keep the discount factor. Either way, $\rho_{T}$ inherits its randomness from the single standard normal $Z$, which is what makes the probability calculation below a one-line standardization.


### Cumulative Probability of a Trade Exit
The scaled NPV is now continuous because $S_T$ is continuous. As in L4a, this is a scheduled terminal event; a take-profit or stop-loss monitored before $T$ is a different first-passage problem.

Let $\rho_\star>-1$. Because $S_T>0$ under GBM, a target at or below $-1$ is cleared with probability one. For a feasible target, the strict event can be written as:
$$
\rho_T>\rho_\star
\quad\Longleftrightarrow\quad
\ln\left(\frac{S_T}{S_0}\right)>\ln(1+\rho_\star)+g_yT.
$$
The GBM log price ratio is normal with mean $(\mu-\sigma^2/2)T$ and standard deviation $\sigma\sqrt T$.

> __Closed-form terminal target probability:__
>
> Standardizing the log-price inequality gives:
> $$
> \mathbb P(\rho_T>\rho_\star)
> =1-\Phi\!\left(
> \frac{\ln(1+\rho_\star)+g_yT-(\mu-\sigma^2/2)T}
> {\sigma\sqrt T}
> \right),
> \qquad \rho_\star>-1,
> $$
> where $\Phi$ is the standard normal CDF, $T>0$, and $\sigma>0$. This uses the real-world drift and volatility because the question is a model probability of profit. Replacing the drift with a risk-neutral drift changes the probability measure and answers a derivative-pricing question instead.

The [GBM trade-rule example](CHEME-5660-L4b-Example-GBM-NPV-TradeRule-Fall-2026.ipynb) checks the formula at the median target and plots the exceedance probability across targets.

___

## Optional Advanced Material
The notebooks below extend today's material. They are optional and are not prerequisites for L5a; the [advanced index](advanced/README.md) lists them with a suggested order.

* [▶ From the binomial lattice to GBM](advanced/lattice-limit/CHEME-5660-L4b-Advanced-LatticeToGBM-Fall-2026.ipynb). Calibrate the binomial lattice of L3b and L4a to the GBM drift and volatility, and watch the terminal distribution and the L4a binomial-tail target probability converge to the lognormal and to today's closed form.
* [▶ First-passage rules under GBM](advanced/first-passage/CHEME-5660-L4b-Advanced-FirstPassage-GBM-Fall-2026.ipynb). Compute take-profit and stop-loss first-passage probabilities under GBM (a closed form for one barrier, the L4a absorbing recursion and Monte Carlo for two) and quantify the effect of the monitoring frequency.
* [▶ Drift uncertainty](advanced/drift-uncertainty/CHEME-5660-L4b-Advanced-DriftUncertainty-Fall-2026.ipynb). Show that the drift estimate depends on the calendar span of the data and not on the sampling frequency, and propagate that uncertainty into the target probability.
* [▶ Monte Carlo versus the closed form](advanced/monte-carlo/CHEME-5660-L4b-Advanced-MonteCarlo-TargetProbability-Fall-2026.ipynb). Estimate the target probability by simulation with standard errors, compare the exact one-step transition with the Euler scheme, and reduce variance with antithetic variates.

___

## Summary
In this lecture, we introduced geometric Brownian motion, estimated its parameters from growth-rate data, tested its empirical implications, and derived a closed-form probability for a scheduled trade.

> __Key Takeaways:__
>
> * **The GBM model gives a tractable lognormal price model:** The log price ratio is normal, the price stays positive, and exact grid-point paths use independent normal increments.
> * **Growth, drift, and volatility play different roles:** One-step growth rates estimate $\bar g$ and $\sigma$; the arithmetic drift is $\mu=\bar g+\sigma^2/2$. Log-price regression gives a descriptive slope, but iid regression uncertainty does not apply to its dependent residuals.
> * **A useful baseline can still fail empirical tests:** Constant-parameter GBM misses heavy tails and volatility clustering, captures zero linear autocorrelation, and converts a terminal return target into one standard-normal tail probability.

Next time, we extend GBM to several correlated assets and introduce the covariance structure needed for portfolios.

___

## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team. 

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success. Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.